In [130]:
#F0484 Melbourne Housing Market - Price
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from data_analysis import summarize_categorical, summarize_numeric
from sklearn.preprocessing import StandardScaler

pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)
SEED = 42

filename = '../MELBOURNE_HOUSE_PRICES_LESS.csv'
df = pd.read_csv(filename)
categorical_col = df.select_dtypes(include=['object']).columns.tolist()
numeric_col = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_col.remove('Price')

Dataset issues handling

In [131]:
#1. duplicate rows
print("Duplicate rows in df:", df.duplicated().sum())
df = df.drop_duplicates()

#2. missing values in price col
df = df.dropna(subset=['Price'])
X = df.drop(columns='Price')
y = df['Price']

#3. Invalid values
print("df")
print("Price <= 50000:", (df['Price'] <= 50000).sum())
print("Rooms <= 0:", (df['Rooms'] <= 0).sum())
print("Distance < 0:", (df['Distance'] < 0).sum())
print("Propertycount < 0:", (df['Propertycount'] < 0).sum())

Duplicate rows in df: 2
df
Price <= 50000: 0
Rooms <= 0: 0
Distance < 0: 0
Propertycount < 0: 0


In [132]:
num_train_sum = summarize_numeric(df[numeric_col])
cat_train_sum = summarize_categorical(df[categorical_col])
num_validation_sum = summarize_numeric(df[numeric_col])
cat_validation_sum = summarize_categorical(df[categorical_col])

In [133]:
print(f"Numerical test df: \n{num_train_sum}\n")
print(f"Categorical test df: \n{cat_train_sum}\n")

print(f"Numerical validation df: \n{num_validation_sum}\n")
print(f"Categorical validation df: \n{cat_validation_sum}\n")

Numerical test df: 
          column      N  % missing  cardinality     min      Q1  median       Q3      max         mean          std
0          Rooms  48432        0.0           14     1.0     2.0     3.0      4.0     31.0     3.071688     0.944705
1       Postcode  48432        0.0          221  3000.0  3051.0  3103.0   3163.0   3980.0  3123.211472   125.535986
2  Propertycount  48432        0.0          359    39.0  4280.0  6567.0  10412.0  21650.0  7566.427218  4457.447851
3       Distance  48432        0.0          176     0.0     7.0    11.7     16.7     55.8    12.702948     7.549995

Categorical test df: 
        column      N  % missing  cardinality                   mode  mode freq  mode %                   mode 2  mode 2 freq  mode 2 %
0       Suburb  48432        0.0          370              Reservoir       1067    2.20           Bentleigh East          696      1.44
1      Address  48432        0.0        44739         56 Victoria St          7    0.01          14 North

In [134]:
from sklearn.model_selection import train_test_split
#split into train, validation and test ds
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=SEED)
X_train, X_validation, y_train, y_validation = train_test_split(X_train, y_train, test_size=0.125, random_state=SEED)
#print(f"Train:\n {train_df.head(5)}\nValidation:\n{validation_df.head(5)}\nTest: \n{test_df.head(5)}")

In [135]:
y_train.head()

28839    1845000.0
50129     725000.0
22182    1900000.0
31069     935000.0
46277    1380000.0
Name: Price, dtype: float64

In [136]:
#4. Outliers
Q1 = y_train.quantile(0.25)
Q3 = y_train.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = y_train[(y_train < lower) | (y_train > upper)]
print("Lower bound:", lower)
print("Upper bound:", upper)
print("Number of outliers:", len(outliers))

y_train = y_train[(y_train >= lower) & (y_train <= upper)]
X_train = X_train.loc[y_train.index]

Lower bound: -280000.0
Upper bound: 2120000.0
Number of outliers: 1782


In [137]:
#5. high cardinality
columns_to_drop = ['Address', 'Suburb', 'SellerG', 'Postcode']
X_train = X_train.drop(columns=columns_to_drop)
X_validation = X_validation.drop(columns=columns_to_drop)
X_test = X_test.drop(columns=columns_to_drop)

In [138]:
X_train['Date'] = pd.to_datetime(X_train['Date'], dayfirst=True)
X_train['Age'] = 2026 - X_train['Date'].dt.year
X_test['Date'] = pd.to_datetime(X_test['Date'], dayfirst=True)
X_test['Age'] = 2026 - X_test['Date'].dt.year
X_validation['Date'] = pd.to_datetime(X_validation['Date'], dayfirst=True)
X_validation['Age'] = 2026 - X_validation['Date'].dt.year
X_train = X_train.drop(columns=["Date"])
X_validation = X_validation.drop(columns=["Date"])
X_test = X_test.drop(columns=["Date"])

In [139]:
#update columns
categorical_col = X_train.select_dtypes(include=['object']).columns.tolist()
numeric_col = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_train_sum = summarize_numeric(X_train[numeric_col])
cat_train_sum = summarize_categorical(X_train[categorical_col])
print(f"Numerical train df: \n{num_train_sum}\n")
print(f"Categorical train df: \n{cat_train_sum}\n")

Numerical train df: 
          column      N  % missing  cardinality   min      Q1  median       Q3      max         mean          std
0          Rooms  36357        0.0           12   1.0     2.0     3.0      4.0     31.0     3.017136     0.918723
1  Propertycount  36357        0.0          354  39.0  4242.0  6567.0  10412.0  21650.0  7593.109442  4516.774769
2       Distance  36357        0.0          176   0.0     7.0    12.0     17.2     55.8    12.896160     7.612836

Categorical train df: 
        column      N  % missing  cardinality                   mode  mode freq  mode %                 mode 2  mode 2 freq  mode 2 %
0         Type  36357        0.0            3                      h      25156   69.19                      u         7314     20.12
1       Method  36357        0.0            5                      S      23151   63.68                     SP         5012     13.79
2   Regionname  36357        0.0            8  Northern Metropolitan      10608   29.18  Southern

One hot encoding for cat features

In [140]:
#if 'Price' in numeric_col: numeric_col.remove('Price')
X_train = pd.get_dummies(X_train, columns=categorical_col, drop_first=False)
X_validation = pd.get_dummies(X_validation, columns=categorical_col, drop_first=False).reindex(columns=X_train.columns, fill_value=0)
X_test = pd.get_dummies(X_test, columns=categorical_col, drop_first=False).reindex(columns=X_train.columns, fill_value=0)

Scaling

In [141]:
scaler_X = StandardScaler()
X_train[numeric_col] = scaler_X.fit_transform(X_train[numeric_col])
X_validation[numeric_col] = scaler_X.transform(X_validation[numeric_col])
X_test[numeric_col] = scaler_X.transform(X_test[numeric_col])

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_validation = scaler_y.transform(y_validation.values.reshape(-1, 1)).flatten()
y_test = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()

In [142]:
#convert to numpy
X_tr = X_train.values.astype(np.float32)
X_val = X_validation.values.astype(np.float32)
X_te = X_test.values.astype(np.float32)

y_tr = y_train.reshape(-1, 1).astype(np.float32)
y_val = y_validation.reshape(-1, 1).astype(np.float32)
y_te = y_test.reshape(-1, 1).astype(np.float32)

print(f"X_train shape: {X_tr.shape}")
print(f"y_train shape: {y_tr.shape}")

X_train shape: (36357, 54)
y_train shape: (36357, 1)


Feed forward ann

In [ ]:
class FeedForwardNN:
    def __init__(self, layer_sizes, activation='relu'):
        self.layer_sizes = layer_sizes
        self.activation_name = activation

        self.weights = []
        self.biases = []
        self.init_parameters()

    def init_parameters(self):
        for i in range(len(self.layer_sizes)-1):
            n_in = self.layer_sizes[i]
            n_out = self.layer_sizes[i+1]

            W = np.random.randn(n_in, n_out)
            b = np.zeros(1, n_out)
            self.weights.append(W)
            self.biases.append(b)

    def activate(self, Z):
        return 1/(1+np.exp(-Z))

    def activate_derivative(self, A): #A-output from activation func
        return A * (1.0 - A)
